# Orthomosaic — Part 3: Cloud-Optimised GeoTIFF (COG)

**Convert the color-corrected orthomosaic to COG layout using the GeoBrix `cog_gbx` writer.**

**The GeoBrix `cog_gbx` writer** wraps GDAL's native `driver="COG"` path via rasterio,
producing a spec-valid COG (internal tiles, overview pyramid, IFDs ordered for HTTP
range reads) without an extra rio-cogeo re-encode pass.  The result opens directly in
QGIS, GDAL, and any COG-aware client.

> **Prerequisite.** `02_color_correction` must have written `orthomosaic_corrected.tif`.

> **Runtime.** Runs on **Serverless environment 5**.  The corrected orthomosaic is loaded
> as a single-partition Spark DataFrame for the `cog_gbx` writer.  For very large mosaics
> (> 1 GB) consider a tiled COG pipeline instead.

---

**Last Update:** September 24, 2026

## Setup

In [ ]:
%run ./config_nb

## Step 1: Generate COG using the GeoBrix `cog_gbx` writer

In [ ]:
import os
import time as _t_cog
from databricks.labs.gbx import pyrx as _pyrx

_t0 = _t_cog.perf_counter()
try:
    from pathlib import Path as _P
    import shutil as _sh
    _groups = discover_groups()
    if not _groups:
        raise RuntimeError(f"no group_* under {output_dir} — run 01/02 first")
    for grp in _groups:
        _gp = group_paths(grp)
        _mani = _pyrx.Manifest(_gp["checkpoint"])
        _src = _gp["corrected"]
        _sig = _pyrx.input_signature({"grp": grp,
                "up": (_mani.get_sig("corrected", grp) or ""),
                "src_size": (os.path.getsize(_src) if os.path.exists(_src) else 0)})
        if _pyrx.checkpoint_skip(_mani, "cog", grp, _sig, force=bool(FORCE_COG)):
            print(f"[cog][skip] group {grp!r} — checkpointed ({_gp['cog']})")
            continue
        _cog_out_dir = _gp["cog_dir"]
        _P(_cog_out_dir).mkdir(parents=True, exist_ok=True)
        # Read the corrected GeoTIFF as driver-local bytes (Volume FUSE). Spark's
        # binaryFile resolves bare paths as dbfs:, so a direct byte read + rst_fromcontent
        # writes the COG via the GeoBrix cog_gbx writer.
        _content = _P(_gp["corrected"]).read_bytes()
        (
            spark.createDataFrame([("orthomosaic_cog", _content)], ["source", "content"])
                 .select("source", rx.rst_fromcontent(F.col("content"), F.lit("GTiff")).alias("tile"))
                 .write.format("cog_gbx").mode("overwrite").save(_cog_out_dir)
        )
        # gtiff_gbx names outputs by hash; normalize to the stable per-group cog name.
        _written = sorted(_P(_cog_out_dir).glob("*.tif"))
        if not _written:
            raise RuntimeError(f"group {grp!r}: gtiff_gbx produced no .tif under {_cog_out_dir}")
        _src = next((w for w in _written if w.name == _P(_gp["cog"]).name), _written[0])
        if _src.resolve() != _P(_gp["cog"]).resolve():
            _sh.move(str(_src), _gp["cog"])
        _mani.mark_done("cog", grp, _sig, _gp["cog"])
        print(f"  group {grp!r}: COG → {_gp['cog']}")
    print(f"COG(s) written in {_t_cog.perf_counter()-_t0:.1f}s ({len(_groups)} group(s))")
except Exception as e:
    print(f"[ERROR] COG step failed after {_t_cog.perf_counter()-_t0:.1f}s: {e}")
    raise

## Step 2: Visualise and validate

In [ ]:
vz.plot_cog(group_paths(discover_groups()[0])["cog"])

## Steps performed

1. **COG conversion** — the `cog_gbx` writer (GeoBrix) applied GDAL's `driver="COG"` path with
   DEFLATE compression, 512-px internal tiles, and an AVERAGE overview pyramid.
2. **Preview** — `vz.plot_cog` (GeoBrix VizX) renders the Cloud-Optimized GeoTIFF over a basemap.

**Next:** Part 4 converts the COG to PMTiles for interactive map serving.

**Re-run behaviour:** each group is checkpointed under `_checkpoint.json`. A re-run skips any group whose COG already matches the corrected-input signature. Set `FORCE_COG = True` in `config_nb` to recompute regardless.